In [ ]:
import pickle 
import numpy as np 
import itertools
import pandas as pd
from pathlib import Path 

# Generate all azim x elevation conditions to explore effects
Targets and distractors presented at positions in -20 to 40 degrees elevation and 0 to 350 azimuth (in steps of 10 degrees).   
All signals at 0dB SNR     

### Will use anechoic room


In [12]:
## Get target and distractor azimuths from human stimuli generation parameters 
human_stim_dir = Path("/om/user/imgriff/datasets/human_azim_spotlight_SWC_2024")

azimuths = np.arange(0, 360, 10)
elevations = np.arange(-20,41,10)

all_positions = list(itertools.product(azimuths, elevations))
all_conditions = list(itertools.product(all_positions, all_positions))

# fix SNR and room index to match 
snr = 0
room_ix = 0

# fix SNR and room index to match 

anechoic_dir = Path("/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval")
room_manifest = pd.read_pickle(anechoic_dir / 'manifest_room.pdpkl')

def get_room_str(room_ix):
    return  (anechoic_dir / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (anechoic_dir / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)


# gen possible location configs in array in array 
cond_dict = {ix:{'target_loc': list(target_loc),
                      'distract_loc': list(dist_loc),
                      'snr': snr,
                      'symmetric_distractor': False,  # Single distractor test 
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (target_loc, dist_loc) in
                    enumerate(all_conditions)} 

# Assert keys contiguous
assert np.all(np.array(list(cond_dict.keys())) == np.arange(len(cond_dict)))
print(len(cond_dict))

63504


In [13]:
### get n total jobs for array 
len(cond_dict)//35

1814

In [14]:
with open('binaural_test_manifests/all_target_distractor_pairs.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)